# Day 12: Beginner NLP Text Processing Lab

This lab introduces classical NLP preprocessing using a small text dataset.

Goal: clean text, convert it into numbers, and build a simple sentiment classifier.

This notebook uses only basic Python libraries so beginners can run it easily.

## 1. Import Libraries

In [ ]:
import re
import math
from collections import Counter, defaultdict

## 2. Create a Small Text Dataset

Labels:
- `1` means positive
- `0` means negative

In [ ]:
data = [
    {"text": "This movie was amazing and emotional!", "label": 1},
    {"text": "I loved the story and the acting was great.", "label": 1},
    {"text": "The product quality is excellent and useful.", "label": 1},
    {"text": "The course explanation was clear and helpful.", "label": 1},
    {"text": "I am happy with the service and support.", "label": 1},
    {"text": "This film was boring and too slow.", "label": 0},
    {"text": "I hated the poor acting and weak story.", "label": 0},
    {"text": "The product broke quickly and disappointed me.", "label": 0},
    {"text": "The lesson was confusing and not helpful.", "label": 0},
    {"text": "The service was bad and very slow.", "label": 0},
]

for row in data:
    print(row)

## 3. Clean Text

Cleaning steps:
1. lowercase text,
2. remove punctuation,
3. split into words,
4. remove simple stopwords.

In [ ]:
stopwords = {"the", "was", "and", "is", "a", "an", "i", "with", "this", "too", "very", "me", "am"}


def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stopwords]
    return words

for row in data:
    row["tokens"] = clean_text(row["text"])

for row in data:
    print(row["text"], "->", row["tokens"])

## 4. Build a Vocabulary

Vocabulary means the unique words found in the cleaned dataset.

In [ ]:
vocabulary = sorted({word for row in data for word in row["tokens"]})
word_to_index = {word: index for index, word in enumerate(vocabulary)}

print("Vocabulary size:", len(vocabulary))
print(vocabulary)

## 5. Bag-of-Words Vectors

Bag-of-words stores word counts for each sentence.

In [ ]:
def bag_of_words(tokens):
    counts = Counter(tokens)
    return [counts.get(word, 0) for word in vocabulary]

for row in data:
    row["bow"] = bag_of_words(row["tokens"])

print("Vocabulary:", vocabulary)
print("First sentence:", data[0]["text"])
print("First bag-of-words vector:", data[0]["bow"])

## 6. TF-IDF Vectors

TF-IDF means Term Frequency - Inverse Document Frequency.

Simple meaning: important words get higher values.

In [ ]:
number_of_documents = len(data)
document_frequency = defaultdict(int)

for word in vocabulary:
    for row in data:
        if word in row["tokens"]:
            document_frequency[word] += 1

idf = {
    word: math.log((1 + number_of_documents) / (1 + document_frequency[word])) + 1
    for word in vocabulary
}


def tfidf(tokens):
    counts = Counter(tokens)
    total_words = len(tokens) if tokens else 1
    vector = []
    for word in vocabulary:
        tf = counts.get(word, 0) / total_words
        vector.append(round(tf * idf[word], 4))
    return vector

for row in data:
    row["tfidf"] = tfidf(row["tokens"])

print("First TF-IDF vector:")
print(data[0]["tfidf"])

## 7. Build a Very Simple Sentiment Classifier

This beginner classifier learns which words appear more in positive text and which words appear more in negative text.

In [ ]:
positive_counts = Counter()
negative_counts = Counter()

for row in data:
    if row["label"] == 1:
        positive_counts.update(row["tokens"])
    else:
        negative_counts.update(row["tokens"])

word_sentiment_score = {}
for word in vocabulary:
    word_sentiment_score[word] = positive_counts[word] - negative_counts[word]

print("Word sentiment scores:")
for word, score in sorted(word_sentiment_score.items()):
    print(word, score)

## 8. Predict New Sentences

In [ ]:
def predict_sentiment(text):
    tokens = clean_text(text)
    score = sum(word_sentiment_score.get(word, 0) for word in tokens)
    if score > 0:
        label = "positive"
    elif score < 0:
        label = "negative"
    else:
        label = "neutral"
    return {"text": text, "tokens": tokens, "score": score, "prediction": label}

examples = [
    "The explanation was clear and useful",
    "The movie was slow and boring",
    "The story had acting"
]

for example in examples:
    print(predict_sentiment(example))

## 9. Beginner Revision Questions

1. Why do we lowercase text?
2. What does regex cleaning remove?
3. What is a vocabulary?
4. What is bag-of-words?
5. What is TF-IDF?
6. Why do NLP models need numbers instead of raw text?